# LLM-based Temporal Reasoning for Cognitive Load and Disfluency Analysis in Spontaneous Speech

**Colab demo** — Tarun Saxena (M25AI2135), M.Tech AI, IIT Jodhpur. Supervisor: Prof. Sidharth Ranjan.

In this notebook, I repeat Experiment 2 of my thesis on Switchboard sentences that I did not use in the thesis.
The held-out pool has 6 stretches of 24 consecutive sentences. The four conversations used in Experiment 2
(sw2005, sw2008, sw2010, sw2012) are left out completely, both sides.

This notebook uses the same code as my GitHub repository
(https://github.com/saxetar/mtp-temporal-cognitive-load): the rule, the prompt, the answer reader and the
metrics all come from `demo/method.py`, which is copied from the thesis pipeline (`mtp/s06_llm_temporal.py`).

Steps: 1 load the feature table, 2 build the held-out pool, 3 choose the number of sentences, 4 rule-based labels,
5 LLM call (Groq), 6 rule vs LLM comparison, 7 trajectory plot, 8 reasons for mismatch.

The features only suggest cognitive load. They do not measure it directly, and I make no diagnostic claims.

In [ ]:
%pip install -q groq
!test -d /content/mtp-temporal-cognitive-load || git clone -q https://github.com/saxetar/mtp-temporal-cognitive-load.git /content/mtp-temporal-cognitive-load

## Configuration

`GROQ_API_KEY` is read from Colab Secrets (key icon on the left; turn on notebook access). It is never printed.
Without a key, the notebook uses a saved answer if there is one, and otherwise shows rule labels only.

In [ ]:
import hashlib, json, os, subprocess, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

REPO_DIR = "/content/mtp-temporal-cognitive-load"
OUTPUT_DIR = "/content/drive/MyDrive/switchboard_sentence_level_outputs"
DATA_PATH = OUTPUT_DIR + "/speech_to_narrative_llm_input.csv"
EXP2_PATH = OUTPUT_DIR + "/speech_to_narrative_temporal_evaluation_50_corrected.csv"
DRIVE_CACHE_DIR = OUTPUT_DIR + "/demo_cache"          # my own new answers are saved here
REPO_CACHE_DIR = REPO_DIR + "/demo/cache"             # answers shipped with the repository
POOL_CSV = REPO_DIR + "/demo/demo_data/heldout_pool.csv"

sys.path.insert(0, REPO_DIR + "/demo")
from method import (LABELS, PRIMARY_MODEL, FALLBACK_MODEL, TEMPERATURE, rule_based_load,
                    temporal_prompt, parse_response, overall_trajectory, metrics)


def get_groq_api_key():
    try:
        from google.colab import userdata
        key = userdata.get("GROQ_API_KEY")
        if key and key.strip():
            return key.strip()
    except Exception:
        pass
    return os.environ.get("GROQ_API_KEY", "").strip()


print("GROQ_API_KEY:", "found" if get_groq_api_key() else "not found (cache or rule labels only)")

## Step 1. Load the feature table

This is the NB05 output of the full run (Switchboard and FluencyBank units with their levels).

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

if os.path.isfile(DATA_PATH):
    full_df = pd.read_csv(DATA_PATH)
    print("Loaded:", DATA_PATH)
    print("Rows: {}, columns: {}".format(*full_df.shape))
    print("Units by dataset:", full_df["dataset"].value_counts().to_dict())
else:
    full_df = None
    print("Feature table not found in Drive. I will use the held-out pool saved in the repository.")

## Step 2. Build the held-out pool

I rebuild the pool from the full feature table with the repository script `demo/tools/build_pool.py` (seed 42),
and then check that it is exactly the same as the `heldout_pool.csv` saved in the repository.

In [ ]:
if full_df is not None and os.path.isfile(EXP2_PATH):
    r = subprocess.run([sys.executable, REPO_DIR + "/demo/tools/build_pool.py", OUTPUT_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError("Pool build failed")
    same = subprocess.run(["git", "-C", REPO_DIR, "diff", "--quiet", "--", "demo/demo_data"]).returncode == 0
    print("Rebuilt pool is identical to the repository copy:", same)
else:
    print("Full data not available, so I use the repository copy of the pool.")

pool = pd.read_csv(POOL_CSV)
manifest = json.load(open(REPO_DIR + "/demo/demo_data/pool_manifest.json"))
print("Excluded Experiment 2 conversations:", ", ".join(manifest["excluded_experiment2_conversations"]))
if os.path.isfile(EXP2_PATH):
    exp2 = pd.read_csv(EXP2_PATH)
    print("Pool sentences that are among the 396 of Experiment 2:", len(set(pool.unit_id) & set(exp2.unit_id)))
    print("Pool conversations that are in Experiment 2:", sorted(set(pool.conversation_id) & set(exp2.conversation_id)) or "none")
print("Pool: {} stretches, {} sentences".format(pool.stretch_id.nunique(), len(pool)))
display(pd.DataFrame(manifest["stretch_table"]))

## Step 3. Choose the stretch and the number of sentences

Pick a stretch (1-6) and a number of sentences (4-24). The sentences are taken from the start of the stretch, as in the web demo.

In [ ]:
STRETCH_ID = 1   # 1 to 6
N_SENTENCES = 8  # 4 to 24

assert 1 <= STRETCH_ID <= 6 and 4 <= N_SENTENCES <= 24, "Use a stretch 1-6 and 4-24 sentences"
seq = pool[pool.stretch_id.eq(STRETCH_ID)].sort_values("position").head(N_SENTENCES).copy()
seq["sentence_number"] = range(1, len(seq) + 1)
seq = seq.reset_index(drop=True)
print("Stretch {}: {}, original sentences {}-{}, N = {}".format(
    STRETCH_ID, seq.conversation_side.iloc[0], seq.original_sentence_number.iloc[0],
    seq.original_sentence_number.iloc[-1], len(seq)))
display(seq[["sentence_number", "original_sentence_number", "sentence_text", "duration_sec", "tokens",
             "speech_rate", "pause_ratio", "disfl_rate", "speech_rate_level", "pause_ratio_level",
             "disfl_rate_level"]].round(3))

## Step 4. Rule-based labels

Rule from the thesis (NB06): **high** if the disfluency rate is high and (the pause ratio is high or the speech rate is low);
**low** if the disfluency rate is low, the pause ratio is low and the speech rate is high; **medium** otherwise.
Levels use the Switchboard 25th/75th percentile cuts of the full corpus.

In [ ]:
seq["rule_based_load_level"] = seq.apply(rule_based_load, axis=1)
display(seq[["sentence_number", "sentence_text", "speech_rate_level", "pause_ratio_level",
             "disfl_rate_level", "rule_based_load_level"]])
print("Label counts:", seq.rule_based_load_level.value_counts().reindex(LABELS, fill_value=0).to_dict())

## Step 5. LLM call (Groq)

The prompt is the Experiment 2 template, word for word. Order: `openai/gpt-oss-120b`, then `openai/gpt-oss-20b`
(same key), then a saved answer for exactly this prompt, then rule labels only.

In [ ]:
prompt = temporal_prompt(seq)
prompt_sha = hashlib.sha256(prompt.encode("utf-8")).hexdigest()
cache_name = "stretch{}_n{:02d}.json".format(STRETCH_ID, len(seq))
print(prompt)
print("-" * 72)


def read_cached():
    for d in (REPO_CACHE_DIR, DRIVE_CACHE_DIR):
        p = os.path.join(d, cache_name)
        if os.path.isfile(p):
            data = json.load(open(p))
            if data.get("prompt_sha256") == prompt_sha:
                return data, p
    return None, None


answer, source, model_used = None, "rule labels only", None
key = get_groq_api_key()
if key:
    from groq import Groq
    client = Groq(api_key=key).with_options(max_retries=1, timeout=90.0)
    for model in (PRIMARY_MODEL, FALLBACK_MODEL):
        try:
            t0 = time.time()
            text = client.chat.completions.create(
                model=model, temperature=TEMPERATURE,
                messages=[{"role": "user", "content": prompt}]).choices[0].message.content
            if not text or not text.strip():
                raise ValueError("empty response")
            answer, source, model_used = text, "live call", model
            print("{}: success in {:.1f} s".format(model, time.time() - t0))
            os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
            json.dump({"stretch_id": STRETCH_ID, "n": len(seq), "model": model, "temperature": TEMPERATURE,
                       "prompt_sha256": prompt_sha,
                       "created_utc": time.strftime("%Y-%m-%d %H:%M", time.gmtime()),
                       "labels_parsed": int(parse_response(text, len(seq)).llm_load_level.notna().sum()),
                       "response": text},
                      open(os.path.join(DRIVE_CACHE_DIR, cache_name), "w"), indent=1)
            print("Saved to", os.path.join(DRIVE_CACHE_DIR, cache_name))
            break
        except Exception as err:
            print("{}: failed ({})".format(model, type(err).__name__))
else:
    print("No GROQ_API_KEY: live call skipped.")

if answer is None:
    cached, path = read_cached()
    if cached:
        answer, source, model_used = cached["response"], "cached answer", cached["model"]
        print("Using the cached answer from", path, "(model {}, saved {} UTC)".format(cached["model"], cached["created_utc"]))
    else:
        print("No cached answer for this stretch and length. Continuing with rule labels only.")
print("Source:", source, "| model:", model_used)

## Step 6. Rule vs LLM comparison

Rule labels are the reference. The labels are read with the same pattern as Experiment 2 (`Sentence k: Cognitive load: ...`).

In [ ]:
if answer is None:
    print("No model answer, so there is nothing to compare. Only rule labels are shown.")
    seq["llm_load_level"], seq["llm_explanation"] = None, ""
else:
    parsed = parse_response(answer, len(seq))
    seq = seq.drop(columns=["llm_load_level", "llm_explanation"], errors="ignore").merge(parsed, on="sentence_number")
    ok = seq[seq.llm_load_level.notna()]
    if len(ok) < len(seq):
        print("No readable label for {} sentence(s); they are left out.".format(len(seq) - len(ok)))
    m = metrics(ok.rule_based_load_level, ok.llm_load_level)
    print("Agreement: {}/{} = {:.2f}%".format(m["matching_labels"], m["sentences"], m["agreement_percent"]))
    print("Macro-F1: {:.4f}   Weighted F1: {:.4f}   Cohen's kappa: {}".format(
        m["macro_f1"], m["weighted_f1"], "n/a" if pd.isna(m["cohen_kappa"]) else "{:.4f}".format(m["cohen_kappa"])))
    print("Majority-label baseline: {:.2f}%".format(ok.rule_based_load_level.value_counts().max() / len(ok) * 100))
    print("For reference, the thesis result on 396 sentences: 214/396 = 54.04%, macro-F1 0.5423, kappa 0.3250 "
          "(always-medium baseline 66.92%). A short stretch is a very small sample.")
    display(pd.DataFrame(m["confusion"], index=["Rule: " + l for l in LABELS], columns=["LLM: " + l for l in LABELS]))
    display(m["per_class"].round(4))
    print("Model's overall trajectory:", overall_trajectory(answer) or "(none)")

## Step 7. Trajectory plot

In [ ]:
LEVEL_Y = {"low": 0, "medium": 1, "high": 2}
x = seq["sentence_number"]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5.6), sharex=True)
ax1.plot(x, seq["rule_based_load_level"].map(LEVEL_Y), "o-", color="#1f4e79", label="Rule-based label")
if seq["llm_load_level"].notna().any():
    ax1.plot(x, seq["llm_load_level"].map(LEVEL_Y), "s--", color="#c55a11", label="LLM label")
ax1.set_yticks([0, 1, 2], LABELS); ax1.set_ylim(-0.4, 2.4); ax1.set_ylabel("Cognitive load")
ax1.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False); ax1.grid(alpha=.3)
ax2.plot(x, seq["speech_rate"], "o-", color="#2e7d32", label="Speech rate (words/s)")
ax2b = ax2.twinx()
ax2b.plot(x, seq["pause_ratio"], "^-", color="#6a1b9a", label="Pause ratio")
ax2b.plot(x, seq["disfl_rate"], "d-", color="#ef6c00", label="Disfluency rate")
h1, l1 = ax2.get_legend_handles_labels(); h2, l2 = ax2b.get_legend_handles_labels()
ax2.legend(h1 + h2, l1 + l2, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, frameon=False)
ax2.set_xlabel("Sentence in the stretch"); ax2.set_ylabel("Speech rate"); ax2b.set_ylabel("Ratio / rate")
ax2.set_xticks(list(x)); ax2.grid(alpha=.3)
fig.tight_layout()
plt.show()

## Step 8. Reasons for mismatch

For each sentence where the labels differ: which part of the rule applied, and the model's own explanation.

In [ ]:
def rule_reason(row):
    d, p, s = row["disfl_rate_level"], row["pause_ratio_level"], row["speech_rate_level"]
    if d == "high" and (p == "high" or s == "low"):
        return "high: disfluency high and " + ("pause high" if p == "high" else "speech rate low")
    if d == "low" and p == "low" and s == "high":
        return "low: disfluency low, pause low and speech rate high"
    return "medium: no extreme pattern (speech {}, pause {}, disfluency {})".format(s, p, d)


if seq["llm_load_level"].isna().all():
    print("No LLM labels, so there is nothing to compare.")
else:
    mm = seq[seq.llm_load_level.notna() & seq.llm_load_level.ne(seq.rule_based_load_level)]
    print("{} of {} sentences differ.".format(len(mm), int(seq.llm_load_level.notna().sum())))
    if len(mm):
        display(pd.DataFrame({"#": mm.sentence_number, "transcript": mm.sentence_text.str.slice(0, 60),
                              "rule": mm.apply(rule_reason, axis=1), "LLM": mm.llm_load_level,
                              "LLM explanation": mm.llm_explanation}))